In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_2"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
def ucb(x, gp, kappa = 2.0):
    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
   
    return mu + kappa * sigma

In [ ]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

In [4]:
X_init = data_in
y_init = data_out

y_trans = y_init.copy()

kernel = Matern(length_scale=0.1, length_scale_bounds=(1e-1, 1e2), nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)

gp.fit(X_init, y_trans)

grid_size = 100
x1 = np.linspace(0, 1, grid_size)
x2 = np.linspace(0, 1, grid_size)
X_candidates = np.array([[i, j] for i in x1 for j in x2])

#evaluate acquisition function
acq_values = ucb(X_candidates, gp, kappa=2.5)

#select next data point
next = X_candidates[np.argmax(acq_values)]

print(cf.format_inputdata(next))

0.000000-1.000000
